# Распределение товаров по торговым точкам через роевой интеллект

## Библиотеки

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from time import time

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## Эвристики

Влияние продаж

In [2]:
sales_factor = 0.01

Влияние загруженности

In [3]:
load_factor = 0.005

Сколько итераций считать

In [4]:
iterations_count = 100

Ранняя остановка. Если лучшее решение не менялось на протяжении указанного кол-ва итераций, то останваливаем алгоритм. Значение 0 значит без ранней остановки

In [5]:
early_stopping = 5

Максимальная скорость познавательного самообучения

In [6]:
max_speed_research = 2.05

Максимальная скорость социального самообучения

In [7]:
max_speed_social = 2.05

Коэффициент $\alpha ∈ (0, 1)$, контролирует близость коэффициента сужения к максимуму

In [8]:
alpha = 0.9
shrink = 2 * alpha / (max_speed_research + max_speed_social - 2)

Число частиц

In [9]:
particle_count = 100

## Подгрузка датасета

In [10]:
col_vol = "Объём товара в закрывающемся магазине (V)"
col_maxcap = "Максимальная вместимость товара в магазине (M)"
col_item = "Код товара"
col_store = "Магазин"
col_remain = "Остаток товара в магазине (R)"
col_sales = "Продажи товара в магазине (S)"
col_dist = "Распределение частиц"

In [11]:
items = pd.read_csv(r"./filtered_datasets/items_closing_store.csv", sep=";")
items = items.set_index([col_item, col_store])
items

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239388     95                                            26.0   
239397     38                                            12.0   
239398     38                                            13.0   
239515     38                                            42.0   
239516     38                                            42.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239388     95                            1.500000   
239397     38                            1.116279   
239398     38                            2.023256   
239515     38                            1.796296   
239516     38                            1.740741   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                    8   
           56                                                    8   
           57                                                    8   
           60                                                    8   
           61                                                    8   
...                                                            ...   
239388     95                                                   25   
239397     38                                                   24   
239398     38                                                   24   
239515     38                                                   30   
239516     38                                                   24   

                    Остаток товара в магазине (R)  
Код товара Магазин                                 
226249     54                                16.0  
           56                                22.0  
           57                                19.0  
           60                                28.0  
           61                                22.0  
...                                           ...  
239388     95                                20.0  
239397     38                                21.0  
239398     38                                45.0  
239515     38                                35.0  
239516     38                                28.0  

[413689 rows x 4 columns]

Со всем датасетом работать очень медленно. Гораздо лучше будет сделать несколько матриц

$i$ - товар, $i ∈ [1, N]$, $N$ - кол-во товаров

$j$ - магазин, $j ∈ [1, M]$, $M$ - кол-во магазинов

*closing_volume* - вектор размерности $N×1$, показывает кол-во товара $i$ в закрывающемся магазине

*sales* - матрица размерности $N×M$, показывает продажи товара $i$ в магазине $j$

*maxcap* - матрица размерности $N×M$, показывает вместимость на полках товара $i$ в магазине $j$

*remains* - матрица размерности $N×M$, показывает остатки товара $i$ в магазине $j$

*pheromones* - матрица размерности $N×M$, показывает феромоны для распределения товара $i$ в магазин $j$

Если товар $i$ не числится в магазине $j$, то тогда соответственные значения $(i, j)$ во всех матрицах равны 0

In [12]:
items_id = items.index.get_level_values(0).unique().values
N = items_id.shape[0]
N

5366

In [13]:
stores_id = items.index.get_level_values(1).unique().values
M = stores_id.shape[0]
M

98

Переиндексация для создания пар $(i, j)$

In [14]:
full_index = pd.MultiIndex.from_product([items_id, stores_id], names=[col_item, col_store])
df_full = items.reindex(full_index)
df_full

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239221     88                                             NaN   
           105                                            NaN   
           29                                             NaN   
           27                                             NaN   
           113                                           17.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239221     88                                 NaN   
           105                                NaN   
           29                                 NaN   
           27                                 NaN   
           113                           0.302326   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                  8.0   
           56                                                  8.0   
           57                                                  8.0   
           60                                                  8.0   
           61                                                  8.0   
...                                                            ...   
239221     88                                                  NaN   
           105                                                 NaN   
           29                                                  NaN   
           27                                                  NaN   
           113                                                10.0   

                    Остаток товара в магазине (R)  
Код товара Магазин                                 
226249     54                                16.0  
           56                                22.0  
           57                                19.0  
           60                                28.0  
           61                                22.0  
...                                           ...  
239221     88                                 NaN  
           105                                NaN  
           29                                 NaN  
           27                                 NaN  
           113                                5.0  

[525868 rows x 4 columns]

Превращаем всё теперь в матрицы

In [15]:
closing_volume = items.groupby(level=0, sort=False).first()[col_vol].values
sales = df_full[col_sales].values.reshape(N, M)
maxcap = df_full[col_maxcap].values.reshape(N, M)
remains = df_full[col_remain].values.reshape(N, M)

Заполняем пропуски нулями

In [16]:
sales = np.nan_to_num(sales, nan=0)
maxcap = np.nan_to_num(maxcap, nan=0)
remains = np.nan_to_num(remains, nan=0)

## Рой частиц

### Частица

Так как рой частиц работает с действительными числами, а распределение - это только целые числа, то нужно сначала округлить значения, а потом восстановить (т.е. repair)

repair работает следующим образом:

1. Сначала исходное распределение с действительными значениями округляем в меньшую сторону
2. Эта операция может лишить нас товаров, потому находим разницу между тем, сколько товаров распределить можно всего, и сколько распределили мы после округления
3. Затем считаем потерянную дробную часть и сортируем в порядке убывания. Таким образом самые большие потерянные дробные части идут впереди
4. Находим индексы тех распределений, где больше всего было потеряно товаров. В numpy шаг 3 и 4 делаются сразу через argsort
5. Теперь осталось добавить по предмету там, где нужнее всего
6. Если же разница товаров отрицательная, то в шаге 3 сортируем в порядке возрастания, чтобы получить самые большие добавленные части. Потом же просто отнимаем эти товары

In [18]:
def particle_rating(particle_distribution):    
    x, v, s, m, r = particle_distribution, closing_volume, sales, maxcap, remains
    c = np.zeros((N, M), dtype=np.int64)  # само распределение

    for i in range(N):  # repair
        c[i] = np.floor(x[i]).astype(np.int64)  # округление до нижней границы
        item_diff = v[i] - np.sum(c[i])  # сколько товаров таким образом потеряли

        if item_diff > 0:
            fractional = x[i] - c[i]  # дробные части
            indexes = np.argsort(-fractional)  # сортировка по значениям дробных частей в порядке убывания
            c[indexes[:item_diff]] += 1  # докинули товары там, где это важнее всего

        elif item_diff < 0:
            fractional = x[i] - c[i]  # дробные части
            indexes = np.argsort(fractional)  # сортировка по значениям дробных частей в порядке возрастания
            c[indexes[:np.abs(item_diff)]] += 1  # убираем товары оттуда, где это важноо всего

    sales_rating = sales_factor * s / (1 + r)
    load_rating = np.divide(load_factor, m, out=np.zeros((N, M), dtype=np.float64), where=m > 0)  # чтобы не поделить на 0
    total_rating = np.sum((c + r) * (sales_rating - load_rating))  # оценка решения
    
    return total_rating, c

### Рой частиц

In [19]:
def particle_swarm_optimization():
    # Здесь индексы None делают здесь из одномерного массива трёхмерные особым образом, разберём на примере
    # Было a = [1, 2, 3]. Сделали b = a[None, :, None]. Получили b = [[[1], [2], [3]]]. Теперь a[0] == b[0][0][0], a[1] == b[0][0][1] и т.д.
    
    particle_distribution = np.random.rand(particle_count, N, M)  # случайные значения от 0 до 1 для начального распределения
    particle_distribution /= particle_distribution.sum(axis=2, keepdims=True)  # потом делаем равномерное распределение, чтобы не превысить кол-во товара в закрывающемся магазине
    particle_distribution *= closing_volume[None, :, None]  # получим таким образом начальное распределение в границах товаров
    
    particle_velocity = np.random.uniform(-closing_volume[None, :, None], closing_volume[None, :, None], size=(particle_count, N, M))  # начальные скорости в пределах [-closing_volume, +closing_volume]
    
    particle_best = particle_distribution.copy()  # изначально, лучшее положение каждой частицы - это её начальное положение 
    
    neighbors = [((i-1) % particle_count, (i+1) % particle_count) for i in range(particle_count)]  # кольцевая топология - два соседа для каждой частицы

    best_rating, best_soluion = None, None
    no_improvements = 0

    for i in range(iterations_count):
        print(f"\nIteration {i+1} out of {iterations_count}")
        ratings, solutions = [], []

        iter_time = time()

        for particle in range(particle_count):
            particle_output = particle_rating(particle_distribution[particle])
            ratings.append(particle_output[0])
            solutions.append(particle_output[1])

        print(f"{particle_count} particles done, took: {time() - iter_time}")
            
        max_index = np.argmax(ratings)

        current_best_rating = ratings[max_index]
        current_best_solution = solutions[max_index]
        no_improvements += 1

        print("Best rating for the current iteration:", current_best_rating)

        if best_rating is None or best_solution is None or (current_best_rating > best_rating):
            best_rating = current_best_rating
            best_solution = current_best_solution
            no_improvements = 0

        print("Total best rating:", best_rating)

        if early_stopping > 0 and no_improvements >= early_stopping:
            print("Early stopping")
            break
    
    return best_rating, best_solution        

In [20]:
time_start = time()
best_rating, best_solution = particle_swarm_optimization()


Iteration 1 out of 100


100%|██████████| 16/16 [00:21<00:00,  1.37s/it]


Best rating for the current iteration: 48.913449433547044
Total best rating: 48.913449433547044

Iteration 2 out of 100


100%|██████████| 16/16 [00:27<00:00,  1.71s/it]


Best rating for the current iteration: 54.6660481825652
Total best rating: 54.6660481825652

Iteration 3 out of 100


...

Best rating for the current iteration: 58.69931485166377
Total best rating: 58.759392566719235

Iteration 44 out of 100


100%|██████████| 16/16 [00:39<00:00,  2.46s/it]


Best rating for the current iteration: 58.67661146583865
Total best rating: 58.759392566719235

Iteration 45 out of 100


100%|██████████| 16/16 [00:39<00:00,  2.46s/it]

Best rating for the current iteration: 58.69910264111026
Total best rating: 58.759392566719235
Early stopping


In [ ]:
print("Time taken:", time() - time_start)

In [21]:
best_rating

np.float64(58.759392566719235)

In [22]:
best_solution

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5366, 98))

## Выгрузка решения

In [23]:
def output_csv(df, filename, with_index=False):
    df_dirpath = Path(r"./solutions")
    df_dirpath.mkdir(parents=True, exist_ok=True)

    df_filepath = df_dirpath / filename

    with df_filepath.open("w") as f:
        df.to_csv(f, sep=";", lineterminator="\n", index=with_index)

In [24]:
df_solution = pd.DataFrame(best_solution, index=items_id, columns=stores_id).stack()
df_solution

226249  54     0
        56     0
        57     0
        60     0
        61     0
              ..
239221  88     0
        105    0
        29     0
        27     0
        113    0
Length: 525868, dtype: int64

In [25]:
df_full[col_dist] = df_solution.reindex(df_full.index).values
df_full

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239221     88                                             NaN   
           105                                            NaN   
           29                                             NaN   
           27                                             NaN   
           113                                           17.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239221     88                                 NaN   
           105                                NaN   
           29                                 NaN   
           27                                 NaN   
           113                           0.302326   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                  8.0   
           56                                                  8.0   
           57                                                  8.0   
           60                                                  8.0   
           61                                                  8.0   
...                                                            ...   
239221     88                                                  NaN   
           105                                                 NaN   
           29                                                  NaN   
           27                                                  NaN   
           113                                                10.0   

                    Остаток товара в магазине (R)  Распределение муравьёв  
Код товара Магазин                                                         
226249     54                                16.0                       0  
           56                                22.0                       0  
           57                                19.0                       0  
           60                                28.0                       0  
           61                                22.0                       0  
...                                           ...                     ...  
239221     88                                 NaN                       0  
           105                                NaN                       0  
           29                                 NaN                       0  
           27                                 NaN                       0  
           113                                5.0                       0  

[525868 rows x 5 columns]

In [26]:
final_distribution = df_full.dropna()

In [27]:
output_csv(final_distribution, "swarm_solution.csv", with_index=True)